In [1]:
import os
import sys
import pdb
import glob
import json
import h5py
import pickle
import pandas as pd
import numpy as np
import scipy.signal
import sox
import soxr
import soundfile as sf
import multiprocessing
import functools
import itertools
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import IPython.display as ipd

sys.path.append('/om2/user/msaddler/python-packages/msutil')
import util_stimuli
import util_misc
import util_figures

sys.path.append('/nese/mit/group/mcdermott/msaddler/pitchnet_dataset/pitchnetDataset/pitchnetDataset')
from dataset_util import initialize_hdf5_file, write_example_to_hdf5



sys.path.append('/om2/user/msaddler/spatial_audio_pipeline/spatial_audio_util')
import util_audio

# sys.path.append('/om2/user/msaddler/tfauditoryutil/paper_phaselocknet')
# import util

import tqdm
tqdm.tqdm.pandas()




/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "
/bin/sh: sox: command not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    


ImportError in `dataset_util.py` No module named 'pyfftw'


## Run following three cells to generate speech embedded in texture noise stimuli as used in Saddler phase locking paper

Use snrs -3, -6, and -9 dB.  
Make sure to update manifest pathname to match desired SNR:
* `/om2/user/msaddler/data_word_recognition/speech_in_synthetic_textures/snrN03/manifest.pdpkl`
* `/om2/user/msaddler/data_word_recognition/speech_in_synthetic_textures/snrN06/manifest.pdpkl`
* `/om2/user/msaddler/data_word_recognition/speech_in_synthetic_textures/snrN09/manifest.pdpkl`

All source signals taken from Mark's human experiment stimuli part of the spatial audio pipeline (see fn foreground and background paths).

Files will be saved with word int classes mapping to 800 word common voice vocabulary:
```word_2_class = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) ```


In [2]:
fn = '/nese/mit/group/mcdermott/msaddler/data_hearinglossnet/source_libraries/JSIN-Behavioral_all__run_000_O27JOGZFN6ZYPUE3DV74Y5CJ52XW2CGJ.h5'
print(f'Inferring `map_word_to_int` and `map_talker_to_int` from:\n{os.path.basename(fn)}')
data = {}
with h5py.File(fn, 'r') as f:
    for k in util_misc.get_hdf5_dataset_key_list(f):
        if ('sources/signal/' in k) and (len(f[k].shape) < 2):
            k_data = k.replace('sources/signal/', '')
            v_data = f[k][:].tolist()
            if isinstance(v_data[0], bytes):
                v_data = [_.decode() for _ in v_data]
            data[k_data] = v_data
            data['split'] = 'eval'
df_unique = pd.DataFrame(data).drop_duplicates()
# map_word_to_int = dict(zip(df_unique.word.values, df_unique.word_int.values))
map_talker_to_int = dict(zip(df_unique.speaker.values, df_unique.speaker_int.values))


Inferring `map_word_to_int` and `map_talker_to_int` from:
JSIN-Behavioral_all__run_000_O27JOGZFN6ZYPUE3DV74Y5CJ52XW2CGJ.h5


#### Only need to run this once 

In [3]:
# get word dict for models 
map_word_to_int = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 

In [4]:
fn_foreground = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/manifest.pdpkl'
fn_background = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/synthetic_textures/manifest.pdpkl'

sr = 44100
dur = 2.5  # set to 2.5 to crop edge artifacts post spatialization

df_foreground = pd.read_pickle(fn_foreground)
df_background = pd.read_pickle(fn_background)

def load_src_fn(src_fn):
    y, sr_input = sf.read(src_fn)
    y = soxr.resample(y, sr_input, sr)
    y = util_stimuli.pad_or_trim_to_len(y, n=int(sr*dur), mode='both', kwargs_pad={})
    y = util_stimuli.set_dBSPL(y, 60.0)
    return y

df_foreground['signal'] = df_foreground['src_fn'].progress_map(load_src_fn)
df_background['signal'] = df_background['src_fn'].progress_map(load_src_fn)





100%|██████████| 16168/16168 [02:41<00:00, 99.84it/s] 


In [5]:
df_foreground.head()

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,split,split_int,sr,src_fn,total_file_duration_in_s,word,signal
0,a-c-norman,3.0,3.0,0.0,swc,0.32,2094.94,2094.62,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,above,"[-0.02980580131899945, -0.02834437994193607, -..."
1,jebjoya,3.0,3.0,0.0,swc,0.49,1715.87,1715.38,/scratch2/weka/mcdermott/msaddler/swc/english/...,2793.356190,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,according,"[-0.0034774617249312013, -0.012965694729374927..."
2,caninedoubletake,3.0,3.0,0.0,swc,0.36,169.03,168.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,987.438730,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,across,"[-0.02131655277351665, -0.020422132113032482, ..."
3,karltalk,3.0,3.0,0.0,swc,0.60,2429.51,2428.91,/scratch2/weka/mcdermott/msaddler/swc/english/...,4802.892336,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,action,"[-0.0015807266117985885, -0.001848813613910525..."
4,s-whistler,3.0,3.0,0.0,swc,0.80,1149.87,1149.07,/scratch2/weka/mcdermott/msaddler/swc/english/...,4463.715556,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,activities,"[-0.010179938808014672, -0.010784021842559808,..."


In [6]:
takler_counts = df_foreground.client_id.value_counts()
print(len(takler_counts))

print(len(takler_counts[takler_counts >= 2]))

164
91


In [7]:
# cut foregrounds to keep those with at least 2 examples

df_foreground = df_foreground[df_foreground.client_id.isin(takler_counts[takler_counts >= 2].index)].reset_index(drop=True) 


In [8]:
# get talker cue signals for each foreground. Cue signals are different excerpt said by the same talker
np.random.seed(0)

cue_signals = []
cue_src_ix =  []
for i, row in tqdm.tqdm(df_foreground.iterrows()):
    cue_examps = df_foreground[(df_foreground.client_id == row.client_id) & (df_foreground.src_fn != row.src_fn)]
    cue_eg = cue_examps.sample(1).iloc[0].signal
    # cue_src
    cue_signals.append(cue_eg)

df_foreground['cue_signal'] = cue_signals

303it [00:00, 874.25it/s]


In [10]:
## Listen to audio examples to make sure cue and target match 

ix = 100 

ipd.display(ipd.Audio(df_foreground.iloc[ix].signal, rate=sr, normalize=False))
ipd.display(ipd.Audio(df_foreground.iloc[ix].cue_signal, rate=sr, normalize=False))


### Make set where texture and foreground are stored separately

In [14]:

fn_dst_dir = Path(f"/om/user/imgriff/datasets/speech_in_synthetic_textures/separated_sources")
fn_dst_dir.mkdir(parents=True, exist_ok=True)
fn_manifest = fn_dst_dir / 'manifest.pdpkl'
fn_dst = fn_dst_dir /  'stim.hdf5'


list_df = []
for index_texture in tqdm.tqdm(np.unique(df_background['index_texture'])):
    df = df_foreground.copy()
    df['label_word_int'] = df['word'].map(map_word_to_int)
    df['label_speaker_int'] = df['client_id'].map(map_talker_to_int)
    list_columns = [
        'label_word_int',
        'label_speaker_int',
        'signal',
        'cue_signal',
    ]
    df = df.loc[:, list_columns].reset_index().rename(columns={'index': 'index_foreground'})
    df['dur'] = dur
    df['sr'] = sr
    df_bg = df_background[df_background['index_texture'] == index_texture]
    # take examples that correspond to index_foreground in df 
    df_bg = df_bg[df_bg.index_example.isin(df.index_foreground)]
    # df_bg
    assert np.array_equal(df_bg.index_example.values, df.index_foreground.values)
    list_target = []
    list_background = []
    for itr0 in range(len(df)):
        # get target 
        list_target.append(df.iloc[itr0]['signal'])
        # get distractor 
        list_background.append(df_bg.iloc[itr0]['signal'])
    df['target'] = list_target
    df['distractor'] = list_background
    df['index_texture'] = df_bg['index_texture'].values
    df['index_example'] = df_bg['index_example'].values
    list_df.append(df)
df = pd.concat(list_df)

print(fn_dst)
# assert not os.path.exists(fn_dst)
with h5py.File(fn_dst, 'w') as f:
    for key in sorted(df.columns):
        if np.issubdtype(df[key].dtype, np.number):
            f[key] = df[key].values
    y = df.iloc[0].signal
    f.create_dataset(
        'target',
        dtype=np.float32,
        shape=[len(df), *y.shape])
    f.create_dataset(
        'distractor',
        dtype=np.float32,
        shape=[len(df), *y.shape])
    f.create_dataset(
        'cue_signal',
        dtype=np.float32,
        shape=[len(df), *y.shape])
    for key in f.keys():
        print(key, f[key].shape, f[key].dtype)
    for itr in tqdm.tqdm(range(len(df))):
        dfi = df.iloc[itr]
        f['target'][itr] = dfi.target
        f['distractor'][itr] = dfi.distractor
        f['cue_signal'][itr] = dfi.cue_signal

print(fn_dst)
df = df.drop(columns=['signal', 'target', 'distractor', 'cue_signal'])
df = df.sort_index(axis=1)
df.to_pickle(fn_manifest)
print(fn_manifest)



100%|██████████| 43/43 [00:03<00:00, 13.12it/s]


/om/user/imgriff/datasets/speech_in_synthetic_textures/separated_sources/stim.hdf5
cue_signal (13029, 110250) float32
distractor (13029, 110250) float32
dur (13029,) float64
index_example (13029,) int64
index_foreground (13029,) int64
index_texture (13029,) int64
label_speaker_int (13029,) int64
label_word_int (13029,) int64
sr (13029,) int64
target (13029, 110250) float32


100%|██████████| 13029/13029 [01:05<00:00, 199.36it/s]


/om/user/imgriff/datasets/speech_in_synthetic_textures/separated_sources/stim.hdf5
/om/user/imgriff/datasets/speech_in_synthetic_textures/separated_sources/manifest.pdpkl


In [5]:
fn_dst = "/om/user/imgriff/datasets/speech_in_synthetic_textures/separated_sources/stim.hdf5"
h5_file = h5py.File(fn_dst, 'r')
sr = 44_100
ipd.display(ipd.Audio(h5_file['cue_signal'][0], rate=sr, normalize=False))
ipd.display(ipd.Audio(h5_file['target'][0], rate=sr, normalize=False))
ipd.display(ipd.Audio(h5_file['distractor'][0], rate=sr, normalize=False))

In [6]:
for key in h5_file.keys():
    print(key, h5_file[key].shape, h5_file[key].dtype)

cue_signal (13029, 110250) float32
distractor (13029, 110250) float32
dur (13029,) float64
index_example (13029,) int64
index_foreground (13029,) int64
index_texture (13029,) int64
label_speaker_int (13029,) int64
label_word_int (13029,) int64
sr (13029,) int64
target (13029, 110250) float32


In [1]:
#### Test dataset 

from corpus.speech_and_texture_test import SpeechAndTextureTestSet

In [2]:
ds = SpeechAndTextureTestSet("/om/user/imgriff/datasets/speech_in_synthetic_textures/separated_sources/stim.hdf5",
 label_type="CV", separated_signals=True, symmetric_distractor=True)

In [4]:
cue, target, distractor, distractor2, word, texture = ds[0]

In [11]:

sr=44100
ipd.display(ipd.Audio(distractor, rate=sr, normalize=False))
ipd.display(ipd.Audio(distractor2, rate=sr, normalize=False))

#### Change snr per set

In [37]:
snr = -9   # either -3, -6, -9, 

test_dir = f"snrN{abs(snr):02d}"
fn_dst_dir = Path(f"/om/user/imgriff/datasets/speech_in_synthetic_textures/{test_dir}")
fn_dst_dir.mkdir(parents=True, exist_ok=True)
fn_manifest = fn_dst_dir / 'manifest.pdpkl'
fn_dst = fn_dst_dir /  'stim.hdf5'


list_df = []
for index_texture in tqdm.tqdm(np.unique(df_background['index_texture'])):
    df = df_foreground.copy()
    df['label_word_int'] = df['word'].map(map_word_to_int)
    df['label_speaker_int'] = df['client_id'].map(map_talker_to_int)
    list_columns = [
        'label_word_int',
        'label_speaker_int',
        'signal',
        'cue_signal',
    ]
    df = df.loc[:, list_columns].reset_index().rename(columns={'index': 'index_foreground'})
    df['dur'] = dur
    df['snr'] = snr
    df['sr'] = sr
    df_bg = df_background[df_background['index_texture'] == index_texture]
    # take examples that correspond to index_foreground in df 
    df_bg = df_bg[df_bg.index_example.isin(df.index_foreground)]
    # df_bg
    assert np.array_equal(df_bg.index_example.values, df.index_foreground.values)
    list_signal = []
    for itr0 in range(len(df)):
        y_foreground = df.iloc[itr0]['signal']
        y_background = df_bg.iloc[itr0]['signal']
        y = util_stimuli.combine_signal_and_noise(y_foreground, y_background, snr).astype(np.float32)
        list_signal.append(y)
    df['signal'] = list_signal
    df['index_texture'] = df_bg['index_texture'].values
    df['index_example'] = df_bg['index_example'].values
    list_df.append(df)
df = pd.concat(list_df)

print(fn_dst)
# assert not os.path.exists(fn_dst)
with h5py.File(fn_dst, 'w') as f:
    for key in sorted(df.columns):
        if np.issubdtype(df[key].dtype, np.number):
            f[key] = df[key].values
    y = df.iloc[0].signal
    f.create_dataset(
        'signal',
        dtype=np.float32,
        shape=[len(df), *y.shape])
    f.create_dataset(
        'cue_signal',
        dtype=np.float32,
        shape=[len(df), *y.shape])
    for key in f.keys():
        print(key, f[key].shape, f[key].dtype)
    for itr in tqdm.tqdm(range(len(df))):
        dfi = df.iloc[itr]
        f['signal'][itr] = dfi.signal
        f['cue_signal'][itr] = dfi.cue_signal
print(fn_dst)
if 'signal' in df.columns:
    df = df.drop(columns=['signal'])
    df = df.sort_index(axis=1)
df.to_pickle(fn_manifest)
print(fn_manifest)



100%|██████████| 43/43 [00:16<00:00,  2.57it/s]


/om/user/imgriff/datasets/speech_in_synthetic_textures/snrN09/stim.hdf5
cue_signal (13029, 110250) float32
dur (13029,) float64
index_example (13029,) int64
index_foreground (13029,) int64
index_texture (13029,) int64
label_speaker_int (13029,) int64
label_word_int (13029,) int64
signal (13029, 110250) float32
snr (13029,) int64
sr (13029,) int64


100%|██████████| 13029/13029 [00:44<00:00, 293.54it/s]


/om/user/imgriff/datasets/speech_in_synthetic_textures/snrN09/stim.hdf5
/om/user/imgriff/datasets/speech_in_synthetic_textures/snrN09/manifest.pdpkl
